In [8]:
import zipfile
import shutil
import os

# Clean up any existing 'my_dataset' folder first
if os.path.exists('my_dataset'):
    shutil.rmtree('my_dataset')

zip_path = '/content/archive.zip'

# Mount Google Drive to fetch your archive.zip directly from your Drive
if not os.path.exists(zip_path):
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        # Search for archive.zip in your Drive root and copy it over
        drive_source = '/content/drive/MyDrive/archive.zip'
        if os.path.exists(drive_source):
            shutil.copy(drive_source, zip_path)
            print("Successfully copied archive.zip from Google Drive.")
        else:
            print(f"Could not find '{drive_source}'. Please ensure the file is named 'archive.zip' and placed in the root of your My Drive folder.")
    except Exception as e:
        print(f"An error occurred while mounting or copying: {e}")

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('my_dataset')
    print("Successfully extracted custom dataset from archive.zip.")
else:
    print("Please upload your 'archive.zip' directly to /content/ via the file panel on the left if Google Drive mounting is not used.")

Successfully extracted custom dataset from archive.zip.


In [9]:

import os
import random
import torch
import torch.nn as nn
import torchvision.transforms as T
import numpy as np
from torchvision.datasets import ImageFolder
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_DIR = "my_dataset/caltech-101"
MODEL_DIR = "models"

print("Device:", device)


Device: cpu


In [10]:

# ============================================================
# MODEL ARCHITECTURES
# These match the architectures used in the training notebook.
# ============================================================

class LeNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(6, 16, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class AlexNetLite(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class VGGLite(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class ResNetLite(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.layer1 = nn.Sequential(
            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = x + self.layer1(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


In [11]:

# ============================================================
# DATASET
# ============================================================

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(
        f"Dataset not found at '{DATASET_DIR}'. "
        "Create the my_dataset/caltech-101 folder beside this notebook."
    )

tf = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

data = ImageFolder(DATASET_DIR, transform=tf)
K = len(data.classes)

print(f"Number of classes: {K}")
print("Sample classes:", data.classes[:10])


Number of classes: 102
Sample classes: ['BACKGROUND_Google', 'Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel']


In [12]:

# ============================================================
# LOAD TRAINED WEIGHTS
# ============================================================

MODEL_FILES = {
    "LeNet-5": ("lenet.pth", lambda: LeNet(K)),
    "AlexNet-lite": ("alexnet_lite.pth", lambda: AlexNetLite(K)),
    "VGG-16-lite": ("vgg_lite.pth", lambda: VGGLite(K)),
    "PlacesNet-lite": ("placesnet_lite.pth", lambda: AlexNetLite(365)),
    "ResNet-18-lite": ("resnet_lite.pth", lambda: ResNetLite(K)),
}

# Metrics from the completed training notebook
METRICS = {
    "LeNet-5": {"parameters": 69826, "training_time": 34.69, "test_accuracy": 0.4524},
    "AlexNet-lite": {"parameters": 280518, "training_time": 79.92, "test_accuracy": 0.5514},
    "VGG-16-lite": {"parameters": 292086, "training_time": 160.93, "test_accuracy": 0.5361},
    "PlacesNet-lite": {"parameters": 314445, "training_time": 83.36, "test_accuracy": 0.3288},
    "ResNet-18-lite": {"parameters": 10838, "training_time": 121.78, "test_accuracy": 0.3441},
}

results = {}

for name, (filename, builder) in MODEL_FILES.items():
    path = os.path.join(MODEL_DIR, filename)

    if not os.path.exists(path):
        print(f"WARNING: Missing {path}")
        continue

    model = builder()
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()

    results[name] = {
        **METRICS[name],
        "model_object": model
    }

print(f"\nLoaded {len(results)} / {len(MODEL_FILES)} models.")



Loaded 5 / 5 models.


In [13]:
import os
import zipfile

zip_models_path = '/content/models.zip'

if os.path.exists(zip_models_path):
    print(f"Extracting {zip_models_path} to 'models/' directory...")
    with zipfile.ZipFile(zip_models_path, 'r') as zip_ref:
        zip_ref.extractall('models')
    print("Extraction complete. Available model files:")
    print(os.listdir('models'))
else:
    print("Could not find 'models.zip'. Please upload it to your /content directory.")

Extracting /content/models.zip to 'models/' directory...
Extraction complete. Available model files:
['placesnet_lite.pth', 'vgg_lite.pth', 'alexnet_lite.pth', 'resnet_lite.pth', 'lenet.pth']


In [14]:

# ============================================================
# DASHBOARD
# ============================================================

display(HTML("""
<style>
.main-container {
    background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
    border-radius: 16px;
    padding: 24px;
    color: white;
    font-family: 'Segoe UI', system-ui, sans-serif;
    box-shadow: 0 10px 25px rgba(0,0,0,0.2);
    margin-bottom: 20px;
}
.header-title {
    font-size: 28px;
    font-weight: 700;
    margin: 0 0 10px 0;
    text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
    letter-spacing: 1px;
}
.header-subtitle {
    font-size: 14px;
    opacity: 0.9;
    margin-bottom: 20px;
}
.dashboard-panel {
    background: rgba(255, 255, 255, 0.1);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 20px;
    border: 1px solid rgba(255,255,255,0.2);
}
.metric-badge {
    background: rgba(255, 255, 255, 0.15);
    border-left: 4px solid #00f2fe;
    padding: 10px 18px;
    border-radius: 6px;
    margin: 8px 0;
    font-size: 15px;
    font-weight: 500;
    display: flex;
    justify-content: space-between;
}
.metric-val {
    color: #00f2fe;
    font-family: 'Courier New', monospace;
    font-weight: bold;
}
"""))

if not results:
    raise RuntimeError(
        "No trained models were loaded. Put the .pth files inside the models/ folder."
    )

available_classes = sorted(data.classes)

model_dropdown = widgets.Dropdown(
    options=list(results.keys()),
    value=list(results.keys())[0],
    description="🧠 Model Network:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

class_dropdown = widgets.Dropdown(
    options=available_classes,
    value="Faces" if "Faces" in available_classes else available_classes[0],
    description="📂 Target Class:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

predict_btn = widgets.Button(
    description="🔮 Trigger Random Image Inference",
    button_style="success",
    tooltip="Run inference on a random image",
    icon="magic",
    layout=widgets.Layout(width="350px", height="42px")
)

out = widgets.Output()

def update_dashboard(change=None):
    with out:
        clear_output(wait=True)

        selected_model = model_dropdown.value
        selected_class = class_dropdown.value
        m = results[selected_model]

        dashboard_html = f"""
        <div class='main-container'>
            <div class='header-title'>🎨 Deep Learning Interactive Playboard</div>
            <div class='header-subtitle'>
                Explore performance benchmarks and test real predictions
                across lightweight model architectures.
            </div>

            <div class='dashboard-panel'>
                <h3 style='margin-top:0;color:#00f2fe;'>
                    📊 Live Metrics: {selected_model}
                </h3>

                <div class='metric-badge'>
                    <span>🔢 Total Net Parameters</span>
                    <span class='metric-val'>{m["parameters"]:,}</span>
                </div>

                <div class='metric-badge'>
                    <span>⚡ Training Convergence Duration</span>
                    <span class='metric-val'>{m["training_time"]:.2f}s</span>
                </div>

                <div class='metric-badge'>
                    <span>🎯 Recorded Evaluation Accuracy</span>
                    <span class='metric-val'>{m["test_accuracy"]:.2%}</span>
                </div>
            </div>
        </div>
        """

        display(HTML(dashboard_html))

        target_dir = os.path.join(DATASET_DIR, selected_class)

        if os.path.exists(target_dir):
            valid_images = [
                f for f in os.listdir(target_dir)
                if os.path.isfile(os.path.join(target_dir, f))
            ]

            display(HTML(
                f"<p style='font-size:15px;'>"
                f"📂 Inspecting <b>{selected_class}</b> — "
                f"<b>{len(valid_images)}</b> images available."
                f"</p>"
            ))

def run_prediction(b):
    with out:
        update_dashboard()

        selected_model = model_dropdown.value
        selected_class = class_dropdown.value

        target_dir = os.path.join(DATASET_DIR, selected_class)

        valid_images = [
            f for f in os.listdir(target_dir)
            if os.path.isfile(os.path.join(target_dir, f))
        ]

        if not valid_images:
            print("No images found in the selected class.")
            return

        random_img_name = random.choice(valid_images)
        img_path = os.path.join(target_dir, random_img_name)

        img = Image.open(img_path).convert("RGB")

        fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

        ax[0].imshow(img)
        ax[0].set_title(
            f"Selected Source Image: {random_img_name}",
            fontsize=11,
            fontweight="bold"
        )
        ax[0].axis("off")

        img_tensor = tf(img).unsqueeze(0).to(device)

        model = results[selected_model]["model_object"]
        model.eval()

        with torch.no_grad():
            raw_out = model(img_tensor)

            if raw_out.shape[1] > K and selected_model == "PlacesNet-lite":
                raw_out = raw_out[:, :K]

            probabilities = torch.softmax(raw_out, dim=1)[0]

            top_probs, top_classes = torch.topk(
                probabilities,
                k=min(5, K)
            )

            top_probs = top_probs.cpu().numpy()
            top_class_names = [
                data.classes[idx]
                for idx in top_classes.cpu().numpy()
            ]

        bars = ax[1].barh(
            top_class_names[::-1],
            top_probs[::-1]
        )

        ax[1].set_xlim(0, 1.05)
        ax[1].set_xlabel("Prediction Confidence Score")
        ax[1].set_title(
            "Top Predicted Probability Distribution",
            fontsize=11,
            fontweight="bold"
        )

        for bar in bars:
            width = bar.get_width()
            ax[1].text(
                width + 0.01,
                bar.get_y() + bar.get_height()/2,
                f"{width:.1%}",
                va="center",
                ha="left",
                fontsize=9,
                fontweight="bold"
            )

        plt.tight_layout()
        plt.show()

        print(
            f"Selected class: {selected_class} | "
            f"Model: {selected_model}"
        )
        print(
            f"Top prediction: {top_class_names[np.argmax(top_probs)]} "
            f"({np.max(top_probs):.2%})"
        )

model_dropdown.observe(update_dashboard, names="value")
class_dropdown.observe(update_dashboard, names="value")
predict_btn.on_click(run_prediction)

control_bar = widgets.VBox([
    model_dropdown,
    class_dropdown,
    predict_btn
], layout=widgets.Layout(
    padding="15px",
    border="1px dashed #3498db",
    margin="10px 0"
))

display(control_bar, out)
update_dashboard()


Output()